In [3]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [4]:
# Load dataset
df = pd.read_csv("dataset_med.csv")

In [5]:
print(df.head())

   id   age  gender      country diagnosis_date cancer_stage family_history  \
0   1  64.0    Male       Sweden     2016-04-05      Stage I            Yes   
1   2  50.0  Female  Netherlands     2023-04-20    Stage III            Yes   
2   3  65.0  Female      Hungary     2023-04-05    Stage III            Yes   
3   4  51.0  Female      Belgium     2016-02-05      Stage I             No   
4   5  37.0    Male   Luxembourg     2023-11-29      Stage I             No   

   smoking_status   bmi  cholesterol_level  hypertension  asthma  cirrhosis  \
0  Passive Smoker  29.4              199.0           0.0     0.0        1.0   
1  Passive Smoker  41.2              280.0           1.0     1.0        0.0   
2   Former Smoker  44.0              268.0           1.0     1.0        0.0   
3  Passive Smoker  43.0              241.0           1.0     1.0        0.0   
4  Passive Smoker  19.7              178.0           0.0     0.0        0.0   

   other_cancer treatment_type end_treatment_date 

In [6]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10185 entries, 0 to 10184
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  10185 non-null  int64  
 1   age                 10185 non-null  float64
 2   gender              10185 non-null  object 
 3   country             10185 non-null  object 
 4   diagnosis_date      10184 non-null  object 
 5   cancer_stage        10184 non-null  object 
 6   family_history      10184 non-null  object 
 7   smoking_status      10184 non-null  object 
 8   bmi                 10184 non-null  float64
 9   cholesterol_level   10184 non-null  float64
 10  hypertension        10184 non-null  float64
 11  asthma              10184 non-null  float64
 12  cirrhosis           10184 non-null  float64
 13  other_cancer        10184 non-null  float64
 14  treatment_type      10184 non-null  object 
 15  end_treatment_date  10184 non-null  object 
 16  surv

In [7]:
# -------------------------------
# DATA CLEANING & PREPROCESSING
# -------------------------------

In [8]:
# Drop irrelevant column
df.drop(columns=['id'], inplace=True)

In [9]:
# Convert date columns to datetime
df['diagnosis_date'] = pd.to_datetime(df['diagnosis_date'])
df['end_treatment_date'] = pd.to_datetime(df['end_treatment_date'])

In [10]:
# Create treatment duration feature
df['treatment_duration'] = (df['end_treatment_date'] - df['diagnosis_date']).dt.days

In [11]:
# Drop original date columns
df.drop(columns=['diagnosis_date', 'end_treatment_date'], inplace=True)

In [12]:
binary_cols = ['family_history']

In [13]:
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

In [14]:
# Handle missing values
df.fillna(df.median(numeric_only=True), inplace=True)

In [15]:
# One-hot encoding for categorical features
df = pd.get_dummies(df, columns=[
    'gender', 'country', 'cancer_stage',
    'smoking_status', 'treatment_type'
], drop_first=True)

In [16]:
# Split features and target
X = df.drop('survived', axis=1)
y = df['survived']

In [17]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [18]:
# -----------------------------
# Random Forest Model
# -----------------------------

In [19]:
rf = RandomForestClassifier(n_estimators=100,random_state=84)

In [20]:
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=84)

In [21]:
# Predictions
y_pred = rf.predict(X_test)

In [22]:
# Evaluation
scr = accuracy_score(y_test,y_pred)
print(f'Accuracy: {scr * 100:.2f}%')
print(classification_report(y_test, y_pred))

Accuracy: 77.37%
              precision    recall  f1-score   support

         0.0       0.77      1.00      0.87      1579
         1.0       0.00      0.00      0.00       458

    accuracy                           0.77      2037
   macro avg       0.39      0.50      0.44      2037
weighted avg       0.60      0.77      0.68      2037



USING BEST PARAMETERS

In [23]:
# Define parameter grid
param_dict = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

In [24]:
# Initialize Random Forest
rf1 = RandomForestClassifier(random_state=42)

In [25]:
# GridSearchCV
random_search = RandomizedSearchCV(
    estimator=rf1,
    param_distributions=param_dict,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

In [26]:
# Fit model
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [None, 10, 20, 30],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300]},
                   scoring='accuracy', verbose=2)

In [27]:
# Best model
best_rf = random_search.best_estimator_

In [28]:
# Predictions
y_pred_best = best_rf.predict(X_test)

In [29]:
# Evaluation
print("Best Parameters:\n", random_search.best_params_)

Best Parameters:
 {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': None, 'bootstrap': False}


In [30]:
scr = accuracy_score(y_test,y_pred_best)
print(f'Accuracy: {scr * 100:.2f}%')
print("\nClassification Report:\n", classification_report(y_test, y_pred_best))

Accuracy: 77.52%

Classification Report:
               precision    recall  f1-score   support

         0.0       0.78      1.00      0.87      1579
         1.0       0.00      0.00      0.00       458

    accuracy                           0.78      2037
   macro avg       0.39      0.50      0.44      2037
weighted avg       0.60      0.78      0.68      2037



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
